# The Enhanced Digital Twin
### Stage 1 of the Control Loop: Building a Network That's Hard to Fool

This notebook explains the three mechanisms that make the simulated network realistic enough to be a meaningful test for the forecaster and planner stages that come after it. The full, tested implementation lives in `src/digital_twin.py` — every function and class referenced below is real, importable code, not pseudocode.

> Why this matters: a twin that's too easy (e.g. latency growing linearly with load) would let a trivial controller "solve" the whole project. These three mechanisms are specifically chosen to break that — see `notebooks/00_overview.ipynb`, Section 1.


## 1. The Congestion Cliff

Latency does not grow linearly with load in a real queuing system — it stays low and flat well under capacity, then rises sharply as load approaches capacity. This is modelled as:

$$
\text{Delay}(L, C) = \text{floor} + \text{scale} \cdot \min\!\left(\frac{L}{C}, 1\right)^{11/5} \; + \; \text{overload\_penalty} \cdot \max\!\left(\frac{L}{C} - 1,\; 0\right)
$$

- Below capacity, the $(L/C)^{2.2}$ term produces the "flat, then cliff" shape.
- Above capacity (a genuine overload — e.g. from a bad allocation decision), an extra **linear** penalty is added, so overload is clearly and increasingly worse than merely being near the cliff, not just a plateau.

*Implemented by:* `src/digital_twin.py :: congestion_cliff_delay()`


In [ ]:
import sys
sys.path.insert(0, "../src")

from digital_twin import congestion_cliff_delay
import matplotlib.pyplot as plt
import numpy as np

capacity = 100.0
loads = np.linspace(0, 140, 300)
delays = [congestion_cliff_delay(l, capacity) for l in loads]

plt.figure(figsize=(7, 4))
plt.plot(loads, delays, color='#1f4e8c')
plt.axvline(capacity, color='crimson', linestyle='--', linewidth=1, label='capacity (100 Mbps)')
plt.title('congestion_cliff_delay() -- real function from src/digital_twin.py')
plt.xlabel('load (Mbps)'); plt.ylabel('delay (ms)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"at 70% load: {congestion_cliff_delay(70, capacity):.1f} ms")
print(f"at 95% load: {congestion_cliff_delay(95, capacity):.1f} ms")
print(f"at 110% load (overload): {congestion_cliff_delay(110, capacity):.1f} ms")


## 2. Correlated Wireless Fading

Real wireless channel quality doesn't flicker randomly every timestep — a bad channel tends to *stay* bad for a while. This is modelled as an AR(1) process:

$$
\text{state}_t = \rho \cdot \text{state}_{t-1} + \sqrt{1-\rho^2}\cdot \epsilon_t, \qquad \epsilon_t \sim \mathcal{N}(0,1)
$$

`state` is squashed through a logistic curve into an *effective-capacity fraction* between `min_fraction` and `1.0` — this is what nominal allocation gets multiplied by each timestep. Higher $\rho$ means slower-changing, more correlated fading.

*Implemented by:* `src/digital_twin.py :: CorrelatedFadingChannel`


In [ ]:
from digital_twin import CorrelatedFadingChannel

correlated = CorrelatedFadingChannel(rho=0.95, seed=2)
independent = CorrelatedFadingChannel(rho=0.0, seed=2)  # rho=0 -> pure independent noise, for contrast

correlated_series = [correlated.step() for _ in range(150)]
independent_series = [independent.step() for _ in range(150)]

plt.figure(figsize=(8, 3.5))
plt.plot(correlated_series, label='rho=0.95 (realistic fading)', color='#1f4e8c')
plt.plot(independent_series, label='rho=0.0 (pure noise, for contrast)', color='#c0392b', alpha=0.7)
plt.title('CorrelatedFadingChannel -- effective capacity fraction over time')
plt.xlabel('timestep'); plt.ylabel('effective capacity fraction')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

def avg_step_jump(series):
    return sum(abs(series[i+1]-series[i]) for i in range(len(series)-1)) / (len(series)-1)

print(f"avg step-to-step jump, rho=0.95: {avg_step_jump(correlated_series):.4f}")
print(f"avg step-to-step jump, rho=0.00: {avg_step_jump(independent_series):.4f}")
print("The correlated channel changes far more smoothly -- exactly the 'stays bad for a while' behaviour real fading has.")


## 3. Shared Base-Station Contention

The subtlest of the three mechanisms, and the one a naive per-slice controller is most likely to miss: giving one slice a much larger *share* of the total allocation degrades every slice's *effective* capacity, not just its own — approximating real shared radio/compute resource contention at a base station.

$$
\text{contention\_penalty}(\text{slice}) = \text{contention\_strength} \times \frac{\sum_{\text{other slices}} \text{allocation}}{\sum_{\text{all slices}} \text{allocation}}
$$

$$
\text{effective\_capacity} = \text{nominal\_allocation} \times \text{fading\_fraction} \times (1 - \text{contention\_penalty})
$$

So URLLC's *nominal* allocation can stay fixed while its *effective* capacity quietly drops purely because eMBB's share of the total grew — this is exactly the failure mode described in `notebooks/00_overview.ipynb`, Section 1.

*Implemented by:* logic inside `src/digital_twin.py :: DigitalTwin.step()` (see `_contention_penalty()`)


In [ ]:
from digital_twin import DigitalTwin, SliceTrafficSpec

specs = [
    SliceTrafficSpec("URLLC", base_demand_mbps=20.0, noise_std_mbps=0.0, spike_probability=0.0),
    SliceTrafficSpec("eMBB", base_demand_mbps=20.0, noise_std_mbps=0.0, spike_probability=0.0),
]

# Same URLLC nominal allocation (30 Mbps) in both runs -- only eMBB's share of the total changes.
twin_light_embb = DigitalTwin(specs, fading_rho=0.0, contention_strength=0.25, seed=5)
obs_light = twin_light_embb.step({"URLLC": 30.0, "eMBB": 10.0})

twin_heavy_embb = DigitalTwin(specs, fading_rho=0.0, contention_strength=0.25, seed=5)
obs_heavy = twin_heavy_embb.step({"URLLC": 30.0, "eMBB": 200.0})

print(f"URLLC effective capacity when eMBB is lightly allocated (10 Mbps):  {obs_light['URLLC'].effective_capacity_mbps:.2f} Mbps")
print(f"URLLC effective capacity when eMBB is heavily allocated (200 Mbps): {obs_heavy['URLLC'].effective_capacity_mbps:.2f} Mbps")
print("\nSame URLLC *nominal* allocation (30 Mbps) both times -- the drop is purely from contention.")


## 4. Putting It Together — What a Fixed Allocation Actually Experiences

The plot below runs the full `DigitalTwin` (all three mechanisms active, plus the traffic generator's occasional spikes) for 200 timesteps under a **fixed** allocation, exactly the baseline this project's agentic planner is meant to beat.

The QoS threshold line (10 ms) is crossed only during traffic spikes, not constantly — this is the calibration this project's evaluation (Stage 6) depends on: if the baseline were *always* violating, there would be nothing interesting left for the planner to fix; if it *never* violated, there would be nothing for the planner to improve either.


In [ ]:
specs = [
    SliceTrafficSpec("URLLC", base_demand_mbps=22.0, noise_std_mbps=1.5,
                      spike_probability=0.02, spike_multiplier=1.9, spike_decay=0.65),
    SliceTrafficSpec("eMBB", base_demand_mbps=45.0, noise_std_mbps=6.0,
                      spike_probability=0.02, spike_multiplier=2.2, spike_decay=0.7),
]
twin = DigitalTwin(specs, fading_rho=0.9, fading_min_fraction=0.8, contention_strength=0.12, seed=7)
fixed_allocation = {"URLLC": 45.0, "eMBB": 60.0}

urllc_latency = []
for t in range(200):
    obs = twin.step(fixed_allocation)
    urllc_latency.append(obs["URLLC"].latency_ms)

plt.figure(figsize=(8, 4))
plt.plot(urllc_latency, color='#c0392b', label='URLLC latency (ms)')
plt.axhline(10.0, color='black', linestyle='--', linewidth=1, label='QoS threshold (10 ms)')
plt.title('URLLC latency under a fixed allocation, 200 timesteps')
plt.xlabel('timestep'); plt.ylabel('latency (ms)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

violations = sum(1 for l in urllc_latency if l > 10.0)
print(f"QoS violations: {violations}/200 timesteps ({violations/200:.1%})")
print(f"mean latency: {sum(urllc_latency)/len(urllc_latency):.2f} ms, max latency: {max(urllc_latency):.2f} ms")
print("\nStable most of the time, breaches concentrated exactly at traffic spikes -- this is the gap Stage 4's planner is built to close.")


## 5. Reproducibility

Every stochastic component (`TrafficGenerator`, `CorrelatedFadingChannel`) accepts a `seed`. `DigitalTwin(..., seed=N)` propagates a seed to all of them consistently, so the exact same allocation sequence produces the exact same observations every run — required for a fair static-vs-RL-vs-agentic comparison in Stage 6, where all three controllers must face *identical* traffic.

*Verified by:* `tests/test_digital_twin.py :: test_twin_is_reproducible_with_same_seed` (and 9 other tests — all passing; run with `PYTHONPATH=src pytest tests/test_digital_twin.py -v`).


## 6. What's Implemented Where

| Concept | Function / Class | Tested by |
|---|---|---|
| Congestion cliff | `congestion_cliff_delay()` | 4 tests: floor behaviour, monotonicity, overload penalty, zero-capacity edge case |
| Correlated fading | `CorrelatedFadingChannel` | 2 tests: stays in bounds, is measurably more correlated than white noise |
| Traffic generation | `TrafficGenerator`, `SliceTrafficSpec` | 1 test: non-negative demand |
| Full twin + contention | `DigitalTwin.step()` | 3 tests: returns all slices, contention degrades other slices, reproducible with a fixed seed |

---
*Next: → `03_forecaster.ipynb`*
